# QLoRA PII Token-Classification — Colab Runner

Runs `qlora/train.py` on a Colab GPU runtime. **Runtime → Change runtime type → GPU (T4 is fine).**

Steps:
1. Verify GPU
2. Clone repo + install deps
3. (Optional) HF login for gated models
4. Train
5. Evaluate
6. Download adapter

In [ ]:
!nvidia-smi

In [ ]:
REPO_URL = "https://github.com/angelafoua/qlora.git"
BRANCH = "claude/lucid-thompson-U0oAw"

import os
if not os.path.isdir("/content/qlora"):
    !git clone --branch {BRANCH} {REPO_URL} /content/qlora
%cd /content/qlora

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -U bitsandbytes accelerate peft transformers datasets

In [ ]:
# Optional: log in to Hugging Face (needed if base_model is gated).
# from huggingface_hub import login; login()

In [ ]:
# Sanity check: dataset and config files exist.
import yaml, json, os
cfg = yaml.safe_load(open("qlora/config.yaml"))
print("base_model:", cfg["base_model"])
print("processed_dataset exists:", os.path.isdir(cfg["processed_dataset"]))
print("label_map exists:", os.path.isfile(cfg["label_map"]))

In [ ]:
!python qlora/train.py --config qlora/config.yaml

In [ ]:
!python qlora/evaluate.py --config qlora/config.yaml

In [ ]:
# Zip and download the fine-tuned adapter + metrics.
import os, shutil
from google.colab import files

REPO_DIR = "/content/qlora"
ADAPTER_DIR = os.path.join(REPO_DIR, "qlora/fine_tuned_model")
METRICS = os.path.join(REPO_DIR, "qlora/metrics.json")

if not os.path.isdir(ADAPTER_DIR):
    raise FileNotFoundError(f"Adapter dir not found: {ADAPTER_DIR}. Did train.py finish?")
shutil.make_archive("/content/fine_tuned_model", "zip", ADAPTER_DIR)
files.download("/content/fine_tuned_model.zip")

if os.path.isfile(METRICS):
    files.download(METRICS)
else:
    print(f"WARNING: {METRICS} not found — check that train.py and evaluate.py both completed successfully.")
